# Getting started with ACE generation — graphite

Sample the reference spectrum, generate thermal-scattering ENDF with the selected engine, then use NJOY to create ACE files. `sample_000` is the unperturbed reference.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [1]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [2]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [3]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import parallel_execution
from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.temperature import resolve_leapr_temperature
elif ENGINE == "ncrystal":
    from input_output.ncrystal.core import NCrystalFile
    import tempfile
    import NCrystal as NC

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/input_2.leapr` | NJOY material template. |
| `ncmat_filename` | `C_sg194_pyrolytic_graphite.ncmat` | Built-in NCrystal graphite. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_6-C-012g.jeff` | Neutron evaluation for NJOY processing. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Used for NJOY processing, including ACE generation with either engine. |

Edit paths only if your checkout or NJOY installation uses different locations.

In [4]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_6-C-012g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if not BASE_ENDF_FILE.is_file():
    raise FileNotFoundError(f"Neutron ENDF missing: {BASE_ENDF_FILE}")
if NJOY_EXECUTABLE is None:
    raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

OUTPUT_DIR = PROJECT_ROOT / "tara_output" / f"ace_{ENGINE}" / "beginners"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Neutron ENDF:", BASE_ENDF_FILE)
print("NJOY:", NJOY_EXECUTABLE)
print("Output directory:", OUTPUT_DIR)

Neutron ENDF: /home/aitor/postdoc/tara_preview_V3_5/tests/fixtures/n_6-C-012g.jeff
NJOY: /home/aitor/miniconda3/envs/postdoct/bin/njoy
Output directory: /home/aitor/postdoc/tara_preview_V3_5/tara_output/ace_ncrystal/beginners


In [5]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "input_2.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")

elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "C_sg194_pyrolytic_graphite.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the graphite reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `293.6` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"C"` | Species supplying the reference spectrum. |

In [6]:
TEMPERATURE_K = 293.6
ATOM_LABEL = "C"
print(f"Engine: {ENGINE_LABEL}")
print(f"temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    leapr_template = template

    resolve_leapr_temperature(template, TEMPERATURE_K, warn_on_reuse=False)
    raw_pdos = template.pdos
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    raw_pdos = material.pdos_for_atom(ATOM_LABEL)
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

Engine: NCrystal
temperature: 293.6 K
NCMAT file: /home/aitor/postdoc/tara_preview_V3_5/tests/fixtures/C_sg194_pyrolytic_graphite.ncmat


## 4. Sample the graphite phonon spectrum

The next cell generates three stochastic spectra and retains the unperturbed reference. `TaraConfig` requires an energy shift bound and an amplitude perturbation bound. Set `N_SAMPLES` explicitly because it defaults to zero. All other sampling settings use TARA’s defaults.

**Required sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `1.0` | Maximum phonon energy shift (meV). |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative change in PDOS amplitude (%). |
| `N_SAMPLES` | `3` | Number of stochastic spectra to generate, excluding the reference. |

**Required for ACE:** the neutron evaluation and NJOY executable from Step 2. Outputs are written to `tara_output/ace_<engine>/beginners/`.

In [7]:
MAX_ENERGY_SHIFT_MEV = 1.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3

In [8]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
)

In [9]:
# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum
)

In [10]:
# Generate the stochastic ensemble with the configured sampling batch size.
ensemble = sampler.generate_ensemble()

if ensemble.reference is None:
    raise RuntimeError("TARA did not provide the expected reference spectrum.")
print(f"Ready: one reference plus {len(ensemble)} stochastic spectra")

Ready: one reference plus 3 stochastic spectra


## 5. Convert spectra to thermal-scattering ENDF files

**NJOY**

TARA uses the graphite LEAPR template from Section 2 and runs NJOY LEAPR in one batch to generate and validate four thermal-scattering ENDF files: one for the unperturbed reference and one for each of the three sampled spectra.

**NCrystal**

TARA runs NCrystal in one batch to generate and validate four thermal-scattering ENDF files: one for the unperturbed reference and one for each of the three sampled spectra.

These files are the inputs to the ACE conversion in the next section.

In [11]:
from processing.pipelines.ace_pipeline import UniversalAcePipeline
from processing.core.config import AceGenerationConfig

if ENGINE == "njoy":
    from processing.orchestrators.njoy_batch import NjoyBatchOrchestrator
elif ENGINE == "ncrystal":
    from processing.orchestrators.ncrystal_batch import NcrystalBatchOrchestrator

In [12]:
# Derive the intermediate output location.
endf_dir = OUTPUT_DIR / "endf"

In [13]:
# Generate validated ENDF tapes using the selected batch processor.
sab_options = dict(
    ensemble=ensemble,
    output_dir=str(endf_dir),
    temperature=TEMPERATURE_K,
)

if ENGINE == "njoy":
    batch = NjoyBatchOrchestrator(
        njoy_exec=str(NJOY_EXECUTABLE),
        n_jobs=1,
        )
    sab_options.update(
        base_leapr_file=str(BASE_LEAPR_FILE)
    )
elif ENGINE == "ncrystal":
    batch = NcrystalBatchOrchestrator(
        n_jobs=1,
    )
    sab_options.update(
        base_ncmat_file=str(BASE_NCMAT_FILE),
        atom_label="C",
    )

sab_tapes = batch.generate_ensemble_sab_tapes(**sab_options)

print(f"Generated {len(sab_tapes)} validated ENDF tapes in {endf_dir}")

Generated 4 validated ENDF tapes in /home/aitor/postdoc/tara_preview_V3_5/tara_output/ace_ncrystal/beginners/endf


## 6. Generate ACE files with NJOY

1. **Build the ACE configuration.** `AceGenerationConfig.from_endf()` reads the base neutron evaluation and the thermal-scattering ENDF files in `sab_tapes` to build one shared configuration.
2. **Run NJOY.** `UniversalAcePipeline.run()` uses `sab_tapes` again to supply the reference and sample files for conversion. Each file is converted once.

With `save_njoy_input=True`, the pipeline saves the shared NJOY input as `ace/njoy_ace.inp`. To reuse it for another sample from the same batch, replace `tape70` with that sample’s thermal-scattering ENDF file.

In [14]:
# Derive the graphite ACE configuration from the neutron and thermal ENDF files.
ace_config = AceGenerationConfig.from_endf(
    base_endf_path=str(BASE_ENDF_FILE),
    tsl_endf_files=sab_tapes,
    temperature=TEMPERATURE_K,
)

In [15]:
# Generate ACE libraries for the reference and three samples.
ace_dir = OUTPUT_DIR / "ace"
ace_pipeline = UniversalAcePipeline(
    njoy_exec=str(NJOY_EXECUTABLE)
)

ace_files = ace_pipeline.run(
    sab_tapes=sab_tapes,
    output_dir=str(ace_dir),
    config=ace_config,
    save_xsdir=True,
    save_njoy_input=True
)

print(f"Generated {len(ace_files)} ACE libraries under {ace_dir}")

Generated 4 ACE libraries under /home/aitor/postdoc/tara_preview_V3_5/tara_output/ace_ncrystal/beginners/ace


## 7. Check ACE file sizes

Check that all four expected ACE files exist and are nonempty, then print their sizes.

In [16]:
# Verify that every reported ACE file exists and contains data.
for sample_id, ace_path in ace_files.items():
    path = Path(ace_path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"ACE file is missing or empty: {sample_id} -> {path}")
    print(f"{sample_id:>12}: {path.name} ({path.stat().st_size:,} bytes)")

print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")
print("File existence and size are basic checks, not a physics-level ACE validation.")

  sample_000: sample_000.ace (30,779,339 bytes)
  sample_001: sample_001.ace (29,063,090 bytes)
  sample_002: sample_002.ace (29,489,697 bytes)
  sample_003: sample_003.ace (28,313,456 bytes)

Output directory: /home/aitor/postdoc/tara_preview_V3_5/tara_output/ace_ncrystal/beginners
File existence and size are basic checks, not a physics-level ACE validation.


See [advanced ACE generation](../03_ace_generation/solid.ipynb) for additional sampling and processing settings.